# From Pixels to Neural Networks: Understanding Deep Learning from the Ground Up

## The Problem: Teach a computer to tell handwritten 3s from 7s

Nothing "magical" here. We'll go from the simplest possible approach, then step by step turn it into a real learning model - writing every line of code, understanding every concept.

## 1. How Computers "See" Images

To humans, an image is shapes and edges. But to a computer, **an image is just a grid of numbers**. Each cell is a pixel with a value: 0 is pure white, 255 is pure black, values in between are shades of gray.

The MNIST dataset (Yann LeCun, 1998): each image is 28x28 - that's **784 numbers**, no more, no less.

In [ ]:
!pip install -Uqq fastai
from fastai.vision.all import *
path = untar_data(URLs.MNIST_SAMPLE)
path

`untar_data` auto-downloads and extracts, returning the path. **Golden rule**: never trust what anyone says about the data - always inspect it yourself:

In [ ]:
path.ls()

2 folders: `train` and `valid`.
- **train**: data for learning
- **valid**: data to check if the model truly understands or is just memorizing - the model has never seen this data during training, like a real exam vs. practice problems.

In [ ]:
(path/'train').ls()

Inside `train`, 2 subdirectories: `3` and `7`. Notice - **no file explicitly says which digit it is**. The folder name IS the label. This is a very common convention for image data. Let's see how many images of each:

In [ ]:
threes = (path/'train'/'3').ls().sorted()
sevens = (path/'train'/'7').ls().sorted()
print(f'Number of 3s: {len(threes)}')
print(f'Number of 7s: {len(sevens)}')

### Let's Open an Image

In [ ]:
im3 = Image.open(threes[1])
im3

`Image` comes from PIL - the standard tool for opening images in Python.

Now let's see it as **numbers** - this is what the computer actually sees:

In [ ]:
im3_t = tensor(im3)
print(f'Shape: {im3_t.shape}')
print(f'Min: {im3_t.min()}, Max: {im3_t.max()}')
print('\nTop-left corner (10x10 pixels):')
im3_t[4:14, 4:14]

All numbers near 0 - this is the white background near the corner. Let's try the center, where the digit ink is:

In [ ]:
im3_t[9:19, 9:19]

Much higher values (near 255) - that's the **black ink stroke**.

## 2. The Naive Approach: Comparing to "Ideal Images"

Before real machine learning, let's try the simplest possible approach: **take all 3 images, stack them up, average each pixel** -> get the "ideal 3". Do the same for 7. To classify a new image, see which ideal image it's closer to.

In [ ]:
three_tensors = [tensor(Image.open(o)) for o in threes]
seven_tensors = [tensor(Image.open(o)) for o in sevens]

stacked_threes = torch.stack(three_tensors).float() / 255
stacked_sevens = torch.stack(seven_tensors).float() / 255
print(f'Shape of stacked_threes: {stacked_threes.shape}')

Shape `[6131, 28, 28]` reads: 6131 images, each with 28 rows and 28 columns. This is a **rank-3 tensor** (rank = number of axes).

`/255` converts pixels from [0,255] to [0,1] - a standard convention when doing math with images.

### Computing the Average Image

In [ ]:
mean3 = stacked_threes.mean(0)
mean7 = stacked_sevens.mean(0)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
show_image(mean3, ctx=axes[0], title='Ideal 3')
show_image(mean7, ctx=axes[1], title='Ideal 7');

`.mean(0)` averages along axis 0 (the axis counting images): for each pixel position, average brightness across all 6131 images.

Result: a blurry image. Where writing is consistent, it's dark. Areas of variation are fuzzy. This is the "ideal 3" and "ideal 7".

## 3. Measuring Distance

We need a formula to know whether a new image is closer to the ideal 3 or ideal 7. Use L1 norm (absolute difference, averaged over all pixels):

In [ ]:
def mnist_distance(a, b):
    return (a - b).abs().mean((-1, -2))

a_3 = stacked_threes[1]
print(f'Distance from 3 image #2 to mean3: {mnist_distance(a_3, mean3):.4f}')
print(f'Distance from 3 image #2 to mean7: {mnist_distance(a_3, mean7):.4f}')

Closer to mean3 - as expected (this IS an image of a 3).

### Broadcasting - PyTorch's Hidden Superpower

Now try computing distance for **all 1010 validation images** at once:

In [ ]:
valid_3_tens = torch.stack([tensor(Image.open(o)) for o in (path/'valid'/'3').ls()]).float() / 255
dists = mnist_distance(valid_3_tens, mean3)
print(f'Shape of dists: {dists.shape}  - 1010 results, one per image!')

No Python loops needed. `valid_3_tens` shape `[1010,28,28]` minus `mean3` shape `[28,28]` - they don't match. But PyTorch automatically "pretends" to copy `mean3` 1010 times to match the shape, **without allocating extra memory**. This is **broadcasting** - computation runs at the C/CUDA level, thousands of times faster than pure Python.

## 4. Writing a Classifier and Measuring Accuracy

In [ ]:
def is_3(x):
    return mnist_distance(x, mean3) < mnist_distance(x, mean7)

valid_7_tens = torch.stack([tensor(Image.open(o)) for o in (path/'valid'/'7').ls()]).float() / 255

accuracy_3s = is_3(valid_3_tens).float().mean()
accuracy_7s = (1 - is_3(valid_7_tens).float()).mean()
overall = (accuracy_3s + accuracy_7s) / 2

print(f'Accuracy on 3s: {accuracy_3s:.2%}')
print(f'Accuracy on 7s: {accuracy_7s:.2%}')
print(f'Overall accuracy: {overall:.2%}')

Over 90% - not bad for such a simple approach!

### But is this actually machine learning?

Recall the definition: ML has a mechanism to automatically check how well current parameters perform, then **automatically adjust** those parameters to improve.

Look at what we just did - `mean3` and `mean7` are computed **once**, then fixed forever. Nothing to "improve". This is just a statistical calculation. **Not machine learning.**

## 5. From "Ideal Images" to "Learnable Weights"

Instead of comparing to a fixed ideal image, let's assign **each pixel a weight** - a number saying "how important is this pixel for deciding if it's a 3 or 7."

Example: pixels in the bottom-right corner are almost never dark when writing a 7, but might be dark for an 8. So that pixel's weight should be low for class "7", high for class "8".

Formula: $$\text{score} = \sum_{i=1}^{784} (x_i \cdot w_i) + b$$

If we have a formula like this, we just need to **find the right set of weights `w`**. And to find them, we use the exact same 7-step process we know: random init -> predict -> loss -> gradient -> update -> repeat.

### Prepare Data for the New Model

Images are currently 28x28 matrices (rank-2) - need to "flatten" into a 784-number vector:

In [ ]:
train_x = torch.cat([stacked_threes, stacked_sevens]).view(-1, 28*28)
print(f'Shape of train_x: {train_x.shape}  - each image is now 784 numbers')

`.view(-1, 28*28)` reshapes each image from `28x28` to `784`. `-1` means "figure out the count automatically."

Now we need corresponding labels - 1 for 3, 0 for 7:

In [ ]:
train_y = tensor([1]*len(threes) + [0]*len(sevens)).unsqueeze(1)
print(f'Shape of train_y: {train_y.shape}')

dset = list(zip(train_x, train_y))
x0, y0 = dset[0]
print(f'First sample: x shape {x0.shape}, y = {y0.item()}')

## 6. The Biggest Trap: Why Accuracy Can't Be a Loss Function

Using accuracy as loss sounds reasonable - it's what we actually care about. But think carefully:

**Gradient** = (change in output) / (change in parameter). Accuracy is all-or-nothing - an image is either classified correctly or incorrectly. Tweaking a weight by an infinitesimal amount almost **never** flips any image from correct to incorrect.

-> Numerator always = 0 -> Gradient always = 0 -> The computer is **blind**, doesn't know which direction to move -> Model stays frozen forever.

We need a **smooth** loss, so every tiny weight tweak makes loss nudge up or down a bit.

### Building a Loss Function

For each image:
- If true label = 1 (it's a 3): loss = distance from prediction to 1 -> the closer to 1, the lower the loss
- If true label = 0 (it's a 7): loss = distance from prediction to 0

In [ ]:
def mnist_loss(predictions, targets):
    return torch.where(targets == 1, 1 - predictions, predictions).mean()

`torch.where(a, b, c)` - a parallel if-statement over the entire tensor: where condition `a` is true, take from `b`, otherwise from `c`.

Let's test on a small batch to see that loss reflects prediction quality:

In [ ]:
preds_good  = tensor([0.9, 0.4, 0.8])  # first 2 good, 3rd decent
preds_bad   = tensor([0.9, 0.4, 0.2])  # 3rd is wrong
targets     = tensor([1.0, 0.0, 1.0])

print(f'Loss with good predictions: {mnist_loss(preds_good, targets):.4f}')
print(f'Loss with bad predictions:  {mnist_loss(preds_bad, targets):.4f}')

Lower loss for better predictions - exactly what we want. Crucially: loss changes **smoothly**, unlike the all-or-nothing nature of accuracy.

## 7. Sigmoid - Squeezing Values into [0,1]

Our `mnist_loss` assumes `predictions` are always in [0,1]. But the dot product `x*w+b` can return **any number** (hugely negative, hugely positive). We need a function to squeeze everything into [0,1]. Enter **sigmoid**:

In [ ]:
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

x_vals = torch.linspace(-5, 5, 100)
plt.plot(x_vals, sigmoid(x_vals))
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
plt.axhline(0, color='black', linewidth=0.5)
plt.axhline(1, color='black', linewidth=0.5)
plt.title('Sigmoid: any number -> always in (0,1)')
plt.grid(True, alpha=0.3);

Properties:
- Accepts any number (hugely negative to hugely positive)
- Always returns a value in (0,1)
- **Smooth, always increasing** curve - no corners or jumps -> gradient-descent friendly

Update loss to always go through sigmoid first:

In [ ]:
def mnist_loss(predictions, targets):
    predictions = predictions.sigmoid()
    return torch.where(targets == 1, 1 - predictions, predictions).mean()

### Summary: Loss vs. Metric

- **Metric** (accuracy): for **humans** to understand how good/bad the model is
- **Loss**: for the **computer** to optimize automatically via gradient

They serve two different purposes. Loss is a **compromise** between the real goal (accuracy) and a function that can be optimized by gradient.

## 8. Mini-batches - You Can't Eat the Whole Dataset at Once

12,000+ images in the training set. Two extreme options:
- **Entire dataset**: accurate gradient but painfully slow per step
- **One image at a time**: fast per step but gradient is noisy and unstable

Solution: **mini-batches** - split into small groups. Bonus: GPUs only shine when there's lots of work to do in parallel.

In [ ]:
dl = DataLoader(dset, batch_size=256, shuffle=True)
xb, yb = first(dl)
print(f'Batch: x shape {xb.shape}, y shape {yb.shape}  - 256 images/batch')

## 9. Assembling the Full Training Loop

Initialize random weights for all 784 pixels, plus a **bias** term - a parameter that lets the line "shift up/down" freely, not forced through the origin (just like `y = w*x + b` from high school algebra).

In [ ]:
def init_params(size, std=1.0):
    return (torch.randn(size) * std).requires_grad_()

weights = init_params((28*28, 1))
bias = init_params(1)

### Matrix Multiplication for the Whole Batch at Once

`xb @ weights` computes exactly `w1*x1 + w2*x2 + ... + w784*x784` for **every image in the batch simultaneously** - not a single Python loop. Python loops are extremely slow compared to optimized matrix ops.

In [ ]:
def linear1(xb):
    return xb @ weights + bias

preds = linear1(train_x[:5])
print(f'Prediction shape: {preds.shape}')

### Gradient Computation and One Training Epoch

In [ ]:
def calc_grad(xb, yb, model):
    preds = model(xb)
    loss = mnist_loss(preds, yb)
    loss.backward()

def train_epoch(model, lr, params):
    for xb, yb in dl:
        calc_grad(xb, yb, model)
        for p in params:
            p.data -= p.grad * lr
            p.grad.zero_()

This structure is **identical** to the loop you wrote for the quadratic/rollercoaster problem. The only difference: we added `for xb, yb in dl` to iterate mini-batches. Nothing magical - just familiar building blocks with one extra piece.

### Validation Accuracy Helper

In [ ]:
def batch_accuracy(xb, yb):
    preds = xb.sigmoid()
    correct = (preds > 0.5) == yb
    return correct.float().mean()

def validate_epoch(model):
    accs = [batch_accuracy(model(xb), yb) for xb, yb in valid_dl]
    return round(torch.stack(accs).mean().item(), 4)

In [ ]:
valid_x = torch.cat([valid_3_tens, valid_7_tens]).view(-1, 28*28)
valid_y = tensor([1]*len(valid_3_tens) + [0]*len(valid_7_tens)).unsqueeze(1)
valid_dset = list(zip(valid_x, valid_y))
valid_dl = DataLoader(valid_dset, batch_size=256)

### Train!

In [ ]:
weights = init_params((28*28, 1))
bias = init_params(1)
params = [weights, bias]

print('Epoch | Accuracy')
print('-' * 20)
for i in range(20):
    train_epoch(linear1, lr=1., params=params)
    acc = validate_epoch(linear1)
    print(f'{i+1:5d} | {acc:.4f}')

Accuracy rises each epoch - the model is **actually learning**!

## 10. Using PyTorch Instead of Hand-Rolling

Writing `init_params` + `linear1` every time is tedious. PyTorch packages this up:

In [ ]:
linear_model = nn.Linear(28*28, 1)
print(f'Weights shape: {linear_model.weight.shape}')
print(f'Bias shape:    {linear_model.bias.shape}')

`nn.Linear` does **exactly** what `init_params` + `linear1` did - contains both weights and bias in one clean object.

In [ ]:
def train_epoch_nn(model, lr):
    for xb, yb in dl:
        calc_grad(xb, yb, model)
        for p in model.parameters():
            p.data -= p.grad * lr
            p.grad.zero_()

model = nn.Linear(28*28, 1)
print('Epoch | Accuracy')
print('-' * 20)
for i in range(20):
    train_epoch_nn(model, lr=1.)
    acc = validate_epoch(model)
    print(f'{i+1:5d} | {acc:.4f}')

## 11. The Leap: A Real Neural Network with ReLU

A purely linear model has a fundamental limit: it can only learn **linear relationships** between pixels and results. But handwriting is far more complex.

You might think: "Just stack more linear layers?" - **Wrong.** The composition of two linear functions is always another linear function. Stacking any number of linear layers is mathematically pointless.

The only way to break this limit: insert a **nonlinear** function between layers. The simplest and most common is **ReLU** - replace every negative number with 0, leave positive numbers unchanged.

In [ ]:
x_vals = torch.linspace(-2, 2, 100)
relu_vals = torch.max(tensor(0.0), x_vals)
plt.plot(x_vals, relu_vals)
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.title('ReLU: x < 0 -> 0,  x >= 0 -> x')
plt.grid(True, alpha=0.3);

In [ ]:
def simple_net(xb):
    res = xb @ w1 + b1              # layer 1: 784 -> 30
    res = torch.max(tensor(0.0), res)   # ReLU
    res = res @ w2 + b2              # layer 2: 30 -> 1
    return res

w1 = init_params((28*28, 30))
b1 = init_params(30)
w2 = init_params((30, 1))
b2 = init_params(1)

print(f'w1: {w1.shape}  (784 pixels -> 30 features)')
print(f'w2: {w2.shape}  (30 features -> 1 result)')

`w1` has 30 columns - the first layer produces 30 "features" from the 784 original pixels, each feature being a different combination of pixels. The number 30 is arbitrary: increase for a more complex model, decrease for simplicity.

In [ ]:
w1 = init_params((28*28, 30))
b1 = init_params(30)
w2 = init_params((30, 1))
b2 = init_params(1)
params2 = [w1, b1, w2, b2]

print('Epoch | Accuracy')
print('-' * 20)
for i in range(20):
    train_epoch(simple_net, lr=1., params=params2)
    acc = validate_epoch(simple_net)
    print(f'{i+1:5d} | {acc:.4f}')

### The Almost-Magical Universal Approximation Theorem

This tiny `simple_net` function, with a large enough matrix, **can solve any computable problem** to arbitrary accuracy.

Intuition: any "wiggly" curve can be approximated by a chain of connected straight line segments - the more short segments, the better the approximation. Each neuron (each "column" in `w1`) is one such line segment; ReLU decides which segments are "on" (=0) or "active".

## 12. Going Deeper: 2 Layers vs. 4 Layers

If one nonlinear layer is enough in theory - why do we need deeper models? The answer is **practical performance**: deeper models can learn more complex features with **fewer parameters** -> faster training, less memory.

Using the exact same infrastructure (`dl`, `valid_dl`, `calc_grad`, `batch_accuracy`), just swap the model for an `nn.Sequential` with 4 layers:

In [ ]:
deep_net = nn.Sequential(
    nn.Linear(28*28, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
)

def train_epoch_nn(model, lr):
    for xb, yb in dl:
        calc_grad(xb, yb, model)
        for p in model.parameters():
            p.data -= p.grad * lr
            p.grad.zero_()

print('Epoch | Accuracy (4-layer deep net)')
print('-' * 35)
for i in range(20):
    train_epoch_nn(deep_net, lr=1.)
    acc = validate_epoch(deep_net)
    print(f'{i+1:5d} | {acc:.4f}')

The 4-layer model (128->64->32->1) achieves higher accuracy than the 2-layer model (30->1) from before, even though the total parameter counts are comparable. This is the power of **deep** learning: depth matters more than width.

In the 1990s, researchers were too focused on proving theory (1 nonlinear layer is enough), so few experimented with deeper models - one reason contributing to the "AI winter".

## Summary: The Journey You Just Completed

| Step | Concept | Tool |
|---|---|---|
| 1 | Image = grid of numbers | PIL, tensor |
| 2 | Pixel similarity | `mean()`, L1 distance |
| 3 | Realizing it's not real ML | No parameters to optimize |
| 4 | Learnable weights per pixel | `w*x + b` |
| 5 | Why accuracy can't be loss | Gradient = 0, model is blind |
| 6 | Smooth loss | `mnist_loss` + `sigmoid` |
| 7 | Mini-batches | `DataLoader(batch_size=256)` |
| 8 | Complete training loop | `train_epoch` + gradient descent |
| 9 | Using built-in PyTorch | `nn.Linear(784, 1)` |
| 10 | Real neural network | ReLU between 2 Linear layers |
| 11 | Deeper model | `nn.Sequential` 4 layers -> higher accuracy |

This is the **entire foundation of modern deep learning**. Everything more complex that comes later (CNNs, Transformers, LLMs...) is built on exactly these building blocks you just assembled today.